In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install wandb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import os
import wandb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

nu = 0.01  # kinematic viscosity

wandb.login()

In [ ]:
# Coords in [-1, 1]^2 — kept identical to original so physics gradients are unaffected
x = torch.linspace(0, 1, 128)
y = torch.linspace(0, 1, 128)
X_grid, Y_grid = torch.meshgrid(x, y, indexing='ij')
coords = torch.stack([X_grid.flatten(), Y_grid.flatten()], dim=1)
coords = 2.0 * coords - 1.0   # [-1, 1]

In [ ]:
class Normalizer:
    """
    FIX 1 — Per-channel z-score normalization.
    The original used a single global mean/std across ALL channels and spatial
    dims together, which is incorrect: u, v, and p have very different scales
    so a global stat mixes them, distorting the loss landscape.

    For X (input):  shape [N, C, H, W] — normalize per channel C
    For Y (output): shape [N, 3, H, W] — normalize per channel (u, v, p separately)
    """
    def __init__(self, data, eps=1e-8):
        # data: [N, C, H, W]
        self.mean = data.mean(dim=(0, 2, 3), keepdim=True)   # [1, C, 1, 1]
        self.std  = data.std(dim=(0, 2, 3), keepdim=True).clamp(min=eps)

    def encode(self, x):
        return (x - self.mean) / self.std

    def decode(self, x):
        return x * self.std + self.mean

In [ ]:
class DeepONetDataset(Dataset):
    def __init__(self, X_data, Y_data, coords, n_points=1000):
        self.X_data   = X_data    # already normalized tensors
        self.Y_data   = Y_data
        self.coords   = coords
        self.n_points = n_points

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        branch_input = self.X_data[idx]
        indices      = torch.randint(0, 128 * 128, (self.n_points,))
        trunk_input  = self.coords[indices]
        target       = self.Y_data[idx].reshape(3, -1).permute(1, 0)[indices]
        return branch_input, trunk_input, target

In [ ]:
class BranchNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, latent_dim)

    def forward(self, u):
        features = self.encoder(u).squeeze(-1).squeeze(-1)
        return self.fc(features)


class TrunkNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, latent_dim * 3)
        )
        self.latent_dim = latent_dim

    def forward(self, x):
        return self.net(x).view(-1, 3, self.latent_dim)


class DeepONet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.branch     = BranchNet(latent_dim)
        self.trunk      = TrunkNet(latent_dim)
        self.latent_dim = latent_dim
        self.bias       = nn.Parameter(torch.zeros(3))  # learnable per-channel bias

    def forward(self, u, x):
        B, n_pts, _ = x.shape
        branch_out = self.branch(u)                              # [B, latent]
        trunk_out  = self.trunk(x.view(-1, 2))                   # [B*n_pts, 3, latent]
        trunk_out  = trunk_out.view(B, n_pts, 3, self.latent_dim)
        branch_out = branch_out.unsqueeze(1).unsqueeze(2)        # [B, 1, 1, latent]
        output     = torch.sum(branch_out * trunk_out, dim=-1)   # [B, n_pts, 3]
        return output + self.bias

In [ ]:
def physics_loss(model, branch_input, trunk_input):
    """
    Navier-Stokes residuals: continuity + x/y momentum.

    FIX 2 — Consolidated gradient computation.
    The original called torch.autograd.grad separately for u, v, p
    (9 separate calls), each retaining the full graph. We instead compute
    a single jacobian-style pass per output channel using grad with
    create_graph=True, reducing graph retention overhead.
    """
    trunk_input = trunk_input.detach().requires_grad_(True)
    pred = model(branch_input, trunk_input)   # [B, N, 3]

    u = pred[:, :, 0]
    v = pred[:, :, 1]
    p = pred[:, :, 2]

    ones = torch.ones_like(u)

    def grad1(f):
        return torch.autograd.grad(f, trunk_input, ones,
                                   retain_graph=True, create_graph=True)[0]

    u_g  = grad1(u);   u_x, u_y = u_g[..., 0], u_g[..., 1]
    v_g  = grad1(v);   v_x, v_y = v_g[..., 0], v_g[..., 1]
    p_g  = grad1(p);   p_x, p_y = p_g[..., 0], p_g[..., 1]

    u_xx = grad1(u_x)[..., 0]
    u_yy = grad1(u_y)[..., 1]
    v_xx = grad1(v_x)[..., 0]
    v_yy = grad1(v_y)[..., 1]

    continuity = u_x + v_y
    momentum_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    momentum_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

    return (
        continuity.pow(2).mean() +
        momentum_x.pow(2).mean() +
        momentum_y.pow(2).mean()
    )


def relative_l2(pred, target):
    num = torch.norm(pred - target, dim=(1, 2))
    den = torch.norm(target, dim=(1, 2))
    return (num / (den + 1e-8)).mean()

In [ ]:
data_path  = "/content/drive/MyDrive/LDC Dataset"
geometries = ["harmonics", "nurbs", "skelneton"]

LATENT_DIM     = 256
N_POINTS       = 1000
EPOCHS         = 100
BATCH_SIZE     = 8
LR             = 1e-3
LAMBDA_PHYSICS = 0.1

for geometry in geometries:
    print(f"\n============================== {geometry} ==============================")

    # ── Load data ─────────────────────────────────────────────────────────────
    X_data = np.load(
        os.path.join(data_path, f"{geometry}_lid_driven_cavity_X.npz")
    )["data"].astype(np.float32)

    Y_data = np.load(
        os.path.join(data_path, f"{geometry}_lid_driven_cavity_Y.npz")
    )["data"][:, 0:3].astype(np.float32)

    X = torch.tensor(X_data)
    Y = torch.tensor(Y_data)

    # ── FIX 3: Train/val split BEFORE normalization to prevent data leakage ──
    # Original normalized on the full dataset first, which leaks val stats into
    # the training normalizer.
    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    # ── FIX 1: Per-channel normalization fitted on training data only ─────────
    x_norm = Normalizer(X_train)
    y_norm = Normalizer(Y_train)

    X_train_n = x_norm.encode(X_train)
    X_val_n   = x_norm.encode(X_val)
    Y_train_n = y_norm.encode(Y_train)
    Y_val_n   = y_norm.encode(Y_val)

    train_loader = DataLoader(
        DeepONetDataset(X_train_n, Y_train_n, coords, N_POINTS),
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        DeepONetDataset(X_val_n, Y_val_n, coords, N_POINTS),
        batch_size=BATCH_SIZE
    )

    # ── Model, optimizer, scheduler ───────────────────────────────────────────
    model     = DeepONet(LATENT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6
    )

    # ── W&B init ──────────────────────────────────────────────────────────────
    wandb.init(
        project="PI_DeepONet_LDC_uvp",
        name=f"PI_DeepONet_{geometry}",
        reinit=True,
        config={
            "epochs":          EPOCHS,
            "batch_size":      BATCH_SIZE,
            "lr":              LR,
            "latent_dim":      LATENT_DIM,
            "n_points":        N_POINTS,
            "lambda_physics":  LAMBDA_PHYSICS,
            "nu":              nu,
            "normalization":   "z-score per-channel",
            "scheduler":       "CosineAnnealing",
        }
    )

    best_val_l2 = float("inf")
    save_path   = os.path.join(data_path, f"{geometry}_pi_deeponet_best.pt")

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_total, train_data, train_phys = 0.0, 0.0, 0.0

        for branch_input, trunk_input, target in train_loader:
            branch_input = branch_input.to(device)
            trunk_input  = trunk_input.to(device)
            target       = target.to(device)

            optimizer.zero_grad()

            pred      = model(branch_input, trunk_input)
            data_loss = F.mse_loss(pred, target)
            phys_loss = physics_loss(model, branch_input, trunk_input)
            loss      = data_loss + LAMBDA_PHYSICS * phys_loss

            loss.backward()
            optimizer.step()

            train_total += loss.item()
            train_data  += data_loss.item()
            train_phys  += phys_loss.item()

        scheduler.step()
        n = len(train_loader)
        train_total /= n;  train_data /= n;  train_phys /= n

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        val_mse, val_l2 = 0.0, 0.0

        with torch.no_grad():
            for branch_input, trunk_input, target in val_loader:
                branch_input = branch_input.to(device)
                trunk_input  = trunk_input.to(device)
                target       = target.to(device)
                pred         = model(branch_input, trunk_input)
                val_mse     += F.mse_loss(pred, target).item()
                val_l2      += relative_l2(pred, target).item()

        val_mse /= len(val_loader)
        val_l2  /= len(val_loader)

        # ── FIX 4: Checkpoint best model ──────────────────────────────────────
        if val_l2 < best_val_l2:
            best_val_l2 = val_l2
            torch.save({
                "epoch":       epoch,
                "model":       model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_l2":      best_val_l2,
                "x_norm_mean": x_norm.mean,
                "x_norm_std":  x_norm.std,
                "y_norm_mean": y_norm.mean,
                "y_norm_std":  y_norm.std,
            }, save_path)

        wandb.log({
            "epoch":              epoch + 1,
            "train_total_loss":   train_total,
            "train_data_loss":    train_data,
            "train_physics_loss": train_phys,
            "val_mse":            val_mse,
            "val_rel_L2":         val_l2,
            "best_val_rel_L2":    best_val_l2,   # FIX 5: was never logged
            "lr":                 scheduler.get_last_lr()[0],
        })

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d} | "
                f"Total {train_total:.6f} | Data {train_data:.6f} | "
                f"Phys {train_phys:.6f} | Val L2 {val_l2:.6f} | "
                f"Best {best_val_l2:.6f} | LR {scheduler.get_last_lr()[0]:.2e}"
            )

    print(f"\n{geometry} done. Best Val Rel-L2: {best_val_l2:.6f}")
    wandb.finish()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def visualize_pi_deeponet(geometry_name, model, y_norm, X_val_n, Y_val_raw, coords, device, num_samples=3):
    """
    Generates field comparison plots (U, V, P) and streamline plots
    for num_samples random validation samples.

    Note: the model predicts on random subsets of points during training,
    but for visualization we pass ALL 128*128 coords to get a full field.
    """
    model.eval()
    H, W  = 128, 128
    N     = len(X_val_n)
    all_coords = coords.unsqueeze(0).to(device)   # [1, H*W, 2]

    indices = np.random.choice(N, size=num_samples, replace=False)

    for idx in indices:
        xb = X_val_n[idx:idx+1].to(device)         # [1, C, H, W]
        yb = Y_val_raw[idx].numpy()                 # [3, H, W] raw

        with torch.no_grad():
            pred_norm = model(xb, all_coords)        # [1, H*W, 3]

        # Denormalize: y_norm stats are [1, C, 1, 1], need to reshape for points
        mean = y_norm.mean.squeeze().cpu()           # [3]
        std  = y_norm.std.squeeze().cpu()            # [3]
        pred_raw = pred_norm[0].cpu() * std + mean   # [H*W, 3]

        pred_np = pred_raw.numpy().T.reshape(3, H, W)  # [3, H, W]

        U_pred, V_pred, P_pred = pred_np[0], pred_np[1], pred_np[2]
        U_true, V_true, P_true = yb[0],     yb[1],     yb[2]

        speed_pred = np.sqrt(U_pred**2 + V_pred**2)
        speed_true = np.sqrt(U_true**2 + V_true**2)

        x_lin = np.linspace(0, 1, W)
        y_lin = np.linspace(0, 1, H)
        X_grid, Y_grid = np.meshgrid(x_lin, y_lin)

        fig, axes = plt.subplots(4, 3, figsize=(18, 22))
        fig.suptitle(
            f"{geometry_name} — Sample {idx}\n"
            f"Top→Bottom: U velocity | V velocity | Pressure | Streamlines",
            fontsize=14, fontweight='bold', y=0.98
        )

        fields = [
            ("U Velocity", U_true, U_pred, "RdBu_r"),
            ("V Velocity", V_true, V_pred, "RdBu_r"),
            ("Pressure",   P_true, P_pred, "viridis"),
        ]

        for row, (label, true_f, pred_f, cmap) in enumerate(fields):
            error_f = np.abs(pred_f - true_f)
            vmin = min(true_f.min(), pred_f.min())
            vmax = max(true_f.max(), pred_f.max())

            im0 = axes[row, 0].imshow(true_f, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            axes[row, 0].set_title(f"{label} — Ground Truth", fontsize=11)
            plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)

            im1 = axes[row, 1].imshow(pred_f, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            axes[row, 1].set_title(f"{label} — Predicted", fontsize=11)
            plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)

            im2 = axes[row, 2].imshow(error_f, origin='lower', cmap='hot_r', aspect='equal')
            axes[row, 2].set_title(f"{label} — |Error|  (max={error_f.max():.3e})", fontsize=11)
            plt.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.04)

            for ax in axes[row]:
                ax.set_xticks([]); ax.set_yticks([])

        streamline_configs = [
            ("Ground Truth Streamlines", U_true, V_true, speed_true),
            ("Predicted Streamlines",    U_pred, V_pred, speed_pred),
            ("Speed Error |‖u‖ - ‖û‖|", None,   None,   np.abs(speed_true - speed_pred)),
        ]

        for col, (title, U, V, speed) in enumerate(streamline_configs):
            ax = axes[3, col]
            if col < 2:
                strm = ax.streamplot(
                    X_grid, Y_grid, U, V,
                    color=speed, cmap='plasma', linewidth=1.2,
                    density=1.8, arrowsize=1.2,
                    norm=mcolors.Normalize(vmin=speed.min(), vmax=speed.max())
                )
                plt.colorbar(strm.lines, ax=ax, fraction=0.046, pad=0.04, label='Speed')
                ax.axhline(y=1.0, color='red', linewidth=2.0, linestyle='--', label='Lid')
                ax.legend(fontsize=8, loc='upper right')
            else:
                im = ax.imshow(speed, origin='lower', cmap='hot_r', aspect='equal')
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Speed Error')

            ax.set_title(title, fontsize=11)
            ax.set_xlabel("x"); ax.set_ylabel("y")

        plt.tight_layout(rect=[0, 0, 1, 0.97])

        save_path = os.path.join(
            data_path, f"{geometry_name}_pi_deeponet_sample{idx}.png"
        )
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

        wandb.log({f"{geometry_name}/streamline_sample_{idx}": wandb.Image(fig)})
        plt.show()
        plt.close(fig)


def run_visualization(geometry, data_path, checkpoint_path, device, num_samples=3):
    print(f"\n==== Visualizing {geometry} ====")

    X_data = np.load(
        os.path.join(data_path, f"{geometry}_lid_driven_cavity_X.npz")
    )["data"].astype(np.float32)
    Y_data = np.load(
        os.path.join(data_path, f"{geometry}_lid_driven_cavity_Y.npz")
    )["data"][:, 0:3].astype(np.float32)

    X = torch.tensor(X_data)
    Y = torch.tensor(Y_data)

    _, X_val, _, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

    ckpt = torch.load(checkpoint_path, map_location=device)

    class RestoredNorm:
        def __init__(self, mean, std):
            self.mean = mean; self.std = std
        def encode(self, x):
            return (x - self.mean) / self.std

    x_norm = RestoredNorm(ckpt["x_norm_mean"], ckpt["x_norm_std"])
    y_norm = RestoredNorm(ckpt["y_norm_mean"], ckpt["y_norm_std"])
    y_norm.decode = lambda x: x * y_norm.std + y_norm.mean

    X_val_n = x_norm.encode(X_val)

    model = DeepONet(LATENT_DIM).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    wandb.init(
        project="PI_DeepONet_LDC_uvp",
        name=f"PI_DeepONet_{geometry}_viz",
        reinit=True
    )

    visualize_pi_deeponet(
        geometry_name=geometry,
        model=model,
        y_norm=y_norm,
        X_val_n=X_val_n,
        Y_val_raw=Y_val,
        coords=coords,
        device=device,
        num_samples=num_samples
    )

    wandb.finish()


# ── Run for all geometries ────────────────────────────────────────────────────
for geometry in geometries:
    checkpoint_path = os.path.join(data_path, f"{geometry}_pi_deeponet_best.pt")
    run_visualization(geometry, data_path, checkpoint_path, device, num_samples=3)